# 原始分钟成交额作为活动强度：与Qlib价×量去重

冻结只读2018–2020发现期，比较Futu原件turnover与同完成bar Qlib原价收盘×原价成交量的日内排序，不看收益。前卡同bar隐含VWAP语义已失败；本卡只判断把原字段当“活动强度”是否几乎重复已使用的价量代理，并分成交量档归因越界。

In [1]:
import os,json,hashlib
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';os.environ['MLFLOW_DISABLE_AGENT_HINT']='1'
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
root=Path.cwd();card=root/'research/stage1_v5_raw_turnover_activity_dedup_20261030';plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(root/plan['failed_source_card']/'audit.json')==plan['failed_source_audit_sha256'] and sha(root/plan['semantics_card']/'audit.json')==plan['semantics_audit_sha256']
assert sha(root/plan['source_manifest'])==plan['source_manifest_sha256'] and sha(root/plan['nas_binding'])==plan['nas_binding_sha256'] and sha(root/plan['qlib_provider_uri']/'qlib_store_manifest.json')==plan['qlib_provider_manifest_sha256']
qlib.init(provider_uri=str(root/plan['qlib_provider_uri']),region='us',kernels=1)
raw={}
for symbol,spec in plan['original_parquets'].items():
 path=root/spec['relative_path'];assert sha(path)==spec['sha256']
 x=pd.read_parquet(path,columns=['time_key','turnover','volume','low','high']);x['datetime']=pd.to_datetime(x.time_key)
 raw[symbol]=x.loc[x.datetime.between('2018-10-15 09:31:00','2020-12-31 16:00:00')].set_index('datetime').sort_index()
print('original source identities passed')

[45089:MainThread](2026-10-30 16:25:49,427) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[45089:MainThread](2026-10-30 16:25:49,924) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[45089:MainThread](2026-10-30 16:25:49,925) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_event_guard_fullminute_20261028')}


original source identities passed


In [2]:
bars=D.features(['SOXL','SOXX'],['$raw_close','$raw_volume','$raw_low','$raw_high','$price_valid'],start_time='2018-10-15 09:31:00',end_time='2020-12-31 16:00:00',freq='1min');assert len(bars)==432720
rows=[];bins=[]
for symbol in ('SOXL','SOXX'):
 q=bars.xs(symbol,level='instrument');x=raw[symbol];assert x.index.equals(q.index)
 valid=q['$price_valid'].eq(1)&q['$raw_volume'].gt(0)&x.turnover.gt(0)
 assert np.max(np.abs(x.volume.to_numpy(float)-q['$raw_volume'].to_numpy(float)))==0
 z=pd.DataFrame(index=q.index);z['reported']=x.turnover.astype(float);z['proxy']=q['$raw_close'].astype(float)*q['$raw_volume'].astype(float)
 z['volume']=q['$raw_volume'].astype(float);z['low']=q['$raw_low'].astype(float);z['high']=q['$raw_high'].astype(float)
 z=z.loc[valid].copy();z['trade_date']=z.index.normalize();z['year']=z.index.year
 z['log_reported']=np.log(z.reported);z['log_proxy']=np.log(z.proxy)
 z['reported_demeaned']=z.log_reported-z.groupby('trade_date').log_reported.transform('mean')
 z['proxy_demeaned']=z.log_proxy-z.groupby('trade_date').log_proxy.transform('mean')
 z['price_residual_bp']=10000*(z.log_reported-z.log_proxy)
 z['implied_vwap']=z.reported/z.volume
 z['inside_1bp']=(z.implied_vwap>=z.low*np.exp(-.0001))&(z.implied_vwap<=z.high*np.exp(.0001))
 for year,g in z.groupby('year'):
  rows.append({'symbol':symbol,'year':int(year),'days':int(g.trade_date.nunique()),'bars':len(g),'day_demeaned_spearman':float(g.reported_demeaned.corr(g.proxy_demeaned,method='spearman')),'median_abs_reported_vs_proxy_bp':float(np.median(np.abs(g.price_residual_bp))),'p90_abs_reported_vs_proxy_bp':float(np.quantile(np.abs(g.price_residual_bp),.9)),'inside_ohlc_fraction':float(g.inside_1bp.mean())})
 for name,mask in [('volume_le_100',z.volume.le(100)),('volume_101_1000',z.volume.gt(100)&z.volume.le(1000)),('volume_gt_1000',z.volume.gt(1000))]:
  g=z.loc[mask];bins.append({'symbol':symbol,'volume_bin':name,'bars':len(g),'inside_ohlc_fraction':float(g.inside_1bp.mean()),'median_abs_price_residual_bp':float(np.median(np.abs(g.price_residual_bp)))})
annual=pd.DataFrame(rows).sort_values(['symbol','year']);annual.to_csv(card/'annual_activity_novelty.csv',index=False)
maximum=max(abs(x['day_demeaned_spearman']) for x in rows)
passed=all(abs(x['day_demeaned_spearman'])<plan['minimum_novelty_gate']['maximum_abs_day_demeaned_spearman'] for x in rows)
audit={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'qlib_rows':len(bars),'maximum_abs_day_demeaned_spearman':maximum,'annual':rows,'volume_bin_consistency':bins,'activity_novelty_gate_pass':bool(passed),'original_same_bar_vwap_gate_pass':False,'future_economic_labels_accessed':False,'new_model_fits':0,'new_candidate_accounts':0,'historical_research_passed':False,'decision':'reported_turnover_activity_distinct_source_pass_needs_semantics' if passed else 'reported_turnover_activity_near_duplicate_and_vwap_semantics_failed'}
(card/'audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({'decision':audit['decision'],'max_abs_rank_corr':maximum,'annual':rows,'volume_bins':bins},ensure_ascii=False,indent=2))

{
  "decision": "reported_turnover_activity_near_duplicate_and_vwap_semantics_failed",
  "max_abs_rank_corr": 0.9999999737956189,
  "annual": [
    {
      "symbol": "SOXL",
      "year": 2018,
      "days": 53,
      "bars": 19913,
      "day_demeaned_spearman": 0.9999993373680883,
      "median_abs_reported_vs_proxy_bp": 5.741489427091295,
      "p90_abs_reported_vs_proxy_bp": 21.261886294286814,
      "inside_ohlc_fraction": 0.8728468839451614
    },
    {
      "symbol": "SOXL",
      "year": 2019,
      "days": 252,
      "bars": 96012,
      "day_demeaned_spearman": 0.9999998076886316,
      "median_abs_reported_vs_proxy_bp": 3.15879706141331,
      "p90_abs_reported_vs_proxy_bp": 12.682231413332811,
      "inside_ohlc_fraction": 0.8423113777444486
    },
    {
      "symbol": "SOXL",
      "year": 2020,
      "days": 253,
      "bars": 98143,
      "day_demeaned_spearman": 0.9999990279646196,
      "median_abs_reported_vs_proxy_bp": 5.028621325724458,
      "p90_abs_reported_vs_

## 中文结论

仅在2018–2020原件/Qlib同钟bar，报告turnover与raw_close×raw_volume的逐日去均值秩相关各年两ETF均>0.999999，超过冻结0.995最大重复门；活动强度几乎重复旧价量。虽然报告成交额/量相对收盘有bp级残差，但前卡同barVWAP语义门失败，不能把它当认证的独立交易均价或ETF申赎流。低量bar越界最集中；不在已看结果上设置新量阈值。0新拟合/账户，SOXX Goal未达。